# 05. RAG 파이프라인 구축

**목표:** 지식 베이스를 FAISS 벡터 인덱스로 구축하고, 질의응답 체인을 테스트합니다.

| 단계 | 구성 요소 |
|------|----------|
| 임베딩 | sentence-transformers/all-MiniLM-L6-v2 |
| 벡터 DB | FAISS (faiss-cpu) |
| LLM | OpenAI GPT-3.5 (없으면 폴백 모드) |
| 청크 크기 | 512 chars (overlap 50) |

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
from teen_mind.rag.embeddings import TextEmbedder
from teen_mind.rag.vector_store import VectorStore
from teen_mind.rag.qa_chain import QAChain

print('RAG 모듈 로드 완료')

## 1. 임베딩 모델 로드

In [ ]:
embedder = TextEmbedder()

# 임베딩 테스트
test_texts = [
    "수면 부족은 우울증의 주요 원인입니다.",
    "운동은 스트레스를 줄이는 데 효과적입니다.",
    "소셜미디어 과사용은 불안감을 증가시킵니다.",
]
vecs = embedder.encode(test_texts)
print(f'임베딩 차원: {vecs.shape}')

## 2. 지식 베이스 로드 및 인덱스 구축

In [ ]:
kb_dir = Path('../data/knowledge_base')
txt_files = sorted(kb_dir.glob('*.txt'))
print(f'지식 베이스 파일 수: {len(txt_files)}')

texts, sources = [], []
for path in txt_files:
    with open(path, encoding='utf-8') as f:
        texts.append(f.read())
    sources.append(path.stem)
    print(f'  로드: {path.name} ({len(texts[-1])} chars)')

In [ ]:
# 청크 크기별 실험 (원칙 10번)
for chunk_size in [256, 512, 1024]:
    store = VectorStore(embedder, chunk_size=chunk_size, chunk_overlap=50)
    store.add_documents(texts, sources)
    print(f'chunk_size={chunk_size}: {len(store.documents)}개 청크')

In [ ]:
# 최종 인덱스 구축 (chunk_size=512)
vector_store = VectorStore(embedder, chunk_size=512, chunk_overlap=50)
vector_store.add_documents(texts, sources)

# 저장
vector_store.save('../data/knowledge_base/faiss_index')
print('벡터 인덱스 저장 완료!')

## 3. 유사도 검색 테스트

In [ ]:
test_queries = [
    "How many hours of sleep do teenagers need?",
    "What is the relationship between social media and depression?",
    "How does exercise affect mental health?",
]

for query in test_queries:
    print(f'\n질문: {query}')
    results = vector_store.search(query, k=2)
    for i, r in enumerate(results, 1):
        print(f'  [{i}] {r["metadata"]["source"]} (score: {r["score"]:.4f})')
        print(f'      {r["text"][:150]}...')

## 4. Q&A 체인 테스트

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')

qa = QAChain(vector_store, top_k=3)
print(f'OpenAI 사용 여부: {qa.use_openai}')

In [ ]:
questions = [
    "소셜미디어를 많이 쓰면 우울증이 생기나요?",
    "청소년에게 권장되는 수면 시간은?",
    "운동이 불안을 줄이는 데 도움이 되나요?",
]

for q in questions:
    print(f'\n{'='*50}')
    print(f'Q: {q}')
    result = qa.ask(q)
    print(f'A: {result["answer"][:500]}')
    print(f'출처: {[s["source"] for s in result["sources"]]}')